Before we build the RAG pipeline, let's get familiar with the data we'll use as our knowledge base.

We run these courses every year, and students keep asking the same questions in Slack. We collected those into an FAQ so people can find answers before asking. Some courses have run for five cohorts, so the FAQ grows large and searching it by hand gets tedious. That's exactly the problem our RAG system will solve.

The FAQ data is available as JSON from the DataTalks.Club website. I maintain that site, so I made the data available at a JSON endpoint we can fetch directly.

Let's fetch it:

In [2]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

This returns a list of courses. Each course has a path field that points to its FAQ data.

Let's fetch all the FAQ documents from all courses:

In [3]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

Each entry has:

- id - unique identifier
- course - course slug (e.g., machine-learning-zoomcamp)
- section - which section of the course
- question - the FAQ question
- answer - the FAQ answer

In [4]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [8]:
import json
with open('D:\\llm-zoomcamp-2026-code\\faq_data\\documents.json', 'w') as f:
    json.dump(documents, f)